In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split , GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder ,LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from catboost import CatBoostClassifier, Pool 

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)


# load dataset

In [3]:
df = pd.read_csv("synthetic_health_dataset.csv")
df.head()

,Gender,Smoking,Alcohol_Consumption,Exercise_Frequency,Blood_Pressure,Cholesterol_Level,Stress_Level,Age,BMI,Heart_Rate,Sleep_Hours,Blood_Sugar_Level,Medication_Use,Family_History,Illness
0,Fe__Male__,No,Moderate,Never,0Normal102,Borderline,High,"""90""",16.6,119,3.6,143.6,NaN,No,Yes
1,Other,Yes,NaN,Never,0Normal102,0Normal102,Low,20,29.9,69,9.9,121.8,NaN,Yes,No
2,__Male__,Yes,Heavy,Daily,High,High,Low,52,33.5,54,8.5,107,NaN,Yes,Yes
3,__Male__,Yes,Heavy,Daily,0Normal102,High,Low,15,20.3,72,9.5,92.1,NaN,No,Yes
4,__Male__,No,Moderate,Often,High,High,Medium,60,36.0,58,4.4,113.6,NaN,No,Yes


# DATA CLEANING

In [4]:
df.columns

Index(['Gender', 'Smoking', 'Alcohol_Consumption', 'Exercise_Frequency',
       'Blood_Pressure', 'Cholesterol_Level', 'Stress_Level', 'Age', 'BMI',
       'Heart_Rate', 'Sleep_Hours', 'Blood_Sugar_Level', 'Medication_Use',
       'Family_History', 'Illness'],
      dtype='object')

In [5]:
column_mapping = {
    'Gender': 'Gender',
    'Smoking': 'Smoking',
    'hol_Consump': 'Alcohol_Consumption',
    'rcise_Freque': 'Exercise_Frequency',
    'lood_Pressur': 'Blood_Pressure',
    'olesterol_Lev': 'Cholesterol_Level',
    'Stress_Level': 'Stress_Level',
    'Age': 'Age',
    'BMI': 'BMI',
    'Heart_Rate': 'Heart_Rate',
    'Sleep_Hours': 'Sleep_Hours',
    'od_Sugar_Leve': 'Blood_Sugar_Level',
    'edication_Usa': 'Medication_Usage',
    'amily_Histor': 'Family_History',
    'Illne': 'Illness'
}
df.rename(columns=column_mapping, inplace=True)

In [6]:
df.head()

,Gender,Smoking,Alcohol_Consumption,Exercise_Frequency,Blood_Pressure,Cholesterol_Level,Stress_Level,Age,BMI,Heart_Rate,Sleep_Hours,Blood_Sugar_Level,Medication_Use,Family_History,Illness
0,Fe__Male__,No,Moderate,Never,0Normal102,Borderline,High,"""90""",16.6,119,3.6,143.6,NaN,No,Yes
1,Other,Yes,NaN,Never,0Normal102,0Normal102,Low,20,29.9,69,9.9,121.8,NaN,Yes,No
2,__Male__,Yes,Heavy,Daily,High,High,Low,52,33.5,54,8.5,107,NaN,Yes,Yes
3,__Male__,Yes,Heavy,Daily,0Normal102,High,Low,15,20.3,72,9.5,92.1,NaN,No,Yes
4,__Male__,No,Moderate,Often,High,High,Medium,60,36.0,58,4.4,113.6,NaN,No,Yes


In [7]:
df["Gender"].value_counts()

Gender
__Male__      336
Fe__Male__    326
Other         326
Name: count, dtype: int64

In [8]:
#  Clean string values in 'Gender' 
df['Gender'] = df['Gender'].astype(str).str.replace('__', '', regex=False)
df['Gender'] = df['Gender'].replace({'FeMale': 'Female', 'Male': 'Male', 'Other': 'Other', 'nan': np.nan})

In [15]:
df["Gender"].value_counts()

Gender
Male      336
Female    326
Other     326
Name: count, dtype: int64

In [9]:
df["Blood_Pressure"].value_counts()

Blood_Pressure
0Normal102    342
Low           336
High          312
Name: count, dtype: int64

In [10]:
df["Cholesterol_Level"].value_counts()

Cholesterol_Level
High          342
Borderline    340
0Normal102    311
Name: count, dtype: int64

In [11]:
df["Blood_Pressure"] = df["Blood_Pressure"].replace('0Normal102', 'Normal')
df["Cholesterol_Level"] = df["Cholesterol_Level"].replace('0Normal102', 'Normal')

In [16]:
df.isna().sum()

Gender                  12
Smoking                  7
Alcohol_Consumption    351
Exercise_Frequency       7
Blood_Pressure          10
Cholesterol_Level        7
Stress_Level             5
Age                     10
BMI                      8
Heart_Rate               9
Sleep_Hours              8
Blood_Sugar_Level        3
Medication_Use         345
Family_History           5
Illness                  0
dtype: int64

In [17]:
corrupted_cols = ['Blood_Pressure', 'Cholesterol_Level']
for col in corrupted_cols:
    df[col] = df[col].replace('0Normal102', 'Normal')

#  Clean 'Age' column 

In [18]:
#  Clean 'Age' column 
df['Age'] = df['Age'].astype(str).str.replace('"', '', regex=False).str.replace("'", "", regex=False)
df['Age'] = pd.to_numeric(df['Age'], errors='coerce')

In [19]:
df.head()

,Gender,Smoking,Alcohol_Consumption,Exercise_Frequency,Blood_Pressure,Cholesterol_Level,Stress_Level,Age,BMI,Heart_Rate,Sleep_Hours,Blood_Sugar_Level,Medication_Use,Family_History,Illness
0,Female,No,Moderate,Never,Normal,Borderline,High,90.0,16.6,119,3.6,143.6,NaN,No,Yes
1,Other,Yes,NaN,Never,Normal,Normal,Low,20.0,29.9,69,9.9,121.8,NaN,Yes,No
2,Male,Yes,Heavy,Daily,High,High,Low,52.0,33.5,54,8.5,107,NaN,Yes,Yes
3,Male,Yes,Heavy,Daily,Normal,High,Low,15.0,20.3,72,9.5,92.1,NaN,No,Yes
4,Male,No,Moderate,Often,High,High,Medium,60.0,36.0,58,4.4,113.6,NaN,No,Yes


In [22]:
df["Blood_Pressure"].value_counts()

Blood_Pressure
Normal    342
Low       336
High      312
Name: count, dtype: int64

In [21]:
df["Illness"].value_counts()

Illness
No     510
Yes    490
Name: count, dtype: int64

# check numeric and categorical features

In [39]:
# 1. Fill Categorical NaNs with the most common value (Mode)
categorical_cols = ['Gender', 'Smoking', 'Alcohol_Consumption', 'Exercise_Frequency', 
                    'Blood_Pressure', 'Cholesterol_Level', 'Stress_Level', 
                    'Medication_Usage', 'Family_History']

for col in categorical_cols:
    mode_value = df[col].mode()[0]
    df[col] = df[col].fillna(mode_value)

# 2. Fill Numerical NaNs with the Median value (protects against outliers)
numerical_cols = ['Age', 'BMI', 'Heart_Rate', 'Sleep_Hours', 'Blood_Sugar_Level']

for col in numerical_cols:
    median_value = df[col].median()
    df[col] = df[col].fillna(median_value)

# Verify that there are 0 null values remaining
print("Remaining Null Values:")
print(df.isnull().sum())

KeyError: 'Medication_Usage'